# Exercise 9: Sentiment Classification with an MLP


## Imports

This cell below exists due to compatibility issues between numpy and gensim. Un-comment if you happen to have the same issue:)

In [ ]:
# !pip uninstall -y numpy scipy gensim
# !pip install numpy==1.24.3
# !pip install scipy==1.11.3
# !pip install gensim==4.3.2

In [ ]:
# Core libraries
import os
import io
import string
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Utility data structures
from collections import defaultdict, Counter

# Natural Language Processing (NLP)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim.downloader as api

# Data preprocessing & evaluation
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, precision_recall_curve, auc
)

# Dimensionality reduction
from sklearn.decomposition import TruncatedSVD

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Downloading data from the web
import requests
import gzip

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(device)

## Dataset retrieval

In [ ]:
# Function that downloads and prepared the IMDB Large Movie Review Dataset
def download_movie_reviews_dataset():
    print("Downloading IMDB Large Movie Review Dataset...")
    url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

    # Download the dataset
    response = requests.get(url, stream=True)
    compressed_file = io.BytesIO(response.content)

    # Extract the tar.gz file
    with gzip.open(compressed_file, 'rb') as f_in:
        with open('aclImdb_v1.tar', 'wb') as f_out:
            f_out.write(f_in.read())

    # Extract the tar file
    import tarfile
    with tarfile.open('aclImdb_v1.tar', 'r') as tar:
        tar.extractall()

    # Clean up
    os.remove('aclImdb_v1.tar')

    # Load the dataset
    def load_reviews(directory, label):
        reviews = []
        for filename in os.listdir(directory):
            if filename.endswith('.txt'):
                with open(os.path.join(directory, filename), 'r', encoding='utf-8') as f:
                    reviews.append((f.read(), label))
        return reviews

    # Load positive and negative reviews from both train and test sets
    positive_train = load_reviews('aclImdb/train/pos', 'positive')
    negative_train = load_reviews('aclImdb/train/neg', 'negative')
    positive_test = load_reviews('aclImdb/test/pos', 'positive')
    negative_test = load_reviews('aclImdb/test/neg', 'negative')

    # Combine all data
    all_data = positive_train + negative_train + positive_test + negative_test

    # Convert to DataFrame
    df = pd.DataFrame(all_data, columns=['text', 'sentiment'])

    # Shuffle the data
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Dataset loaded! Total samples: {len(df)}")
    return df

df = download_movie_reviews_dataset()


## Data preprocessing

In [ ]:
# First, split into training and temp (which will be further split into dev and test)
train_data, test_data = train_test_split(df, test_size=0.3, random_state=42, stratify=df['sentiment'])

# Print the sizes to verify the split
print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")

In [ ]:
# Custom tokenizer method that will be used in data preprocessing
def tokenize(data):
  stop_words = stopwords.words('english')
  stop_words.extend(string.punctuation)
  stop_words.extend(["the", "of", "and", "as", "a", "to", "in", "on", "for"])
  tokenized_samples = []
  for sample in data:
    tokens = []
    # Split text into sentences
    sentences = sent_tokenize(sample)
    for sent in sentences:
        # Tokenize each sentence into words
        words = word_tokenize(sent)
        for word in words:
            # Filter out stopwords and unwanted tokens
            if '\n' in word or "\t" in word or "--" in word or "*" in word or "@" in word or "#" in word or word.lower() in stop_words:
                continue
            if word.strip():
                # Process the token and add to list
                tokens.append(word.replace('"', "'").strip().lower())
    tokenized_samples.append(tokens)

  return tokenized_samples

In [ ]:
# Calculate dataset statistics for IMDB movie reviews
def calculate_imdb_stats(data):
    # Calculate number of documents and classes
    num_documents = len(data)
    sentiment_distribution = data['sentiment'].value_counts().to_dict()

    # Calculate document lengths
    doc_lengths = data['text'].apply(len)
    avg_doc_length = doc_lengths.mean()
    min_doc_length = doc_lengths.min()
    max_doc_length = doc_lengths.max()

    # Calculate word counts
    word_counts = data['text'].apply(lambda x: len(x.split()))
    avg_word_count = word_counts.mean()
    min_word_count = word_counts.min()
    max_word_count = word_counts.max()

    # Calculate vocabulary size
    tokenized_texts = data['text'].apply(lambda x: x.lower().split())
    vocabulary = set()
    for tokens in tokenized_texts:
        vocabulary.update(tokens)
    vocab_size = len(vocabulary)

    return {
        "num_documents": num_documents,
        "sentiment_distribution": sentiment_distribution,
        "avg_doc_length": avg_doc_length,
        "min_doc_length": min_doc_length,
        "max_doc_length": max_doc_length,
        "avg_word_count": avg_word_count,
        "min_word_count": min_word_count,
        "max_word_count": max_word_count,
        "vocab_size": vocab_size
    }

# Split data into train, dev and test sets
train_size = 25000  # Original IMDB has 25K training samples
test_size = 25000   # Original IMDB has 25K test samples

# Ensure we have a balanced dataset
train_data = pd.concat([
    df[df['sentiment'] == 'positive'].iloc[:train_size//2],
    df[df['sentiment'] == 'negative'].iloc[:train_size//2]
])
test_data = pd.concat([
    df[df['sentiment'] == 'positive'].iloc[train_size//2:train_size//2+test_size//2],
    df[df['sentiment'] == 'negative'].iloc[train_size//2:train_size//2+test_size//2]
])

# Create a development set from the training set
train_data, dev_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['sentiment'])

# Calculate and display dataset statistics
print("Calculating dataset statistics...")
train_stats = calculate_imdb_stats(train_data)
dev_stats = calculate_imdb_stats(dev_data)
test_stats = calculate_imdb_stats(test_data)

print("Dataset Statistics for IMDB Movie Reviews:\n")
print("Training Set:")
print(f"  Number of documents: {train_stats['num_documents']}")
print(f"  Sentiment distribution: {train_stats['sentiment_distribution']}")
print(f"  Average document length: {train_stats['avg_doc_length']:.2f} characters")
print(f"  Document length range: {train_stats['min_doc_length']} to {train_stats['max_doc_length']} characters")
print(f"  Average word count: {train_stats['avg_word_count']:.2f} words")
print(f"  Word count range: {train_stats['min_word_count']} to {train_stats['max_word_count']} words")
print(f"  Vocabulary size: {train_stats['vocab_size']} unique words")

print("\nDevelopment Set:")
print(f"  Number of documents: {dev_stats['num_documents']}")
print(f"  Sentiment distribution: {dev_stats['sentiment_distribution']}")
print(f"  Average document length: {dev_stats['avg_doc_length']:.2f} characters")
print(f"  Average word count: {dev_stats['avg_word_count']:.2f} words")

print("\nTest Set:")
print(f"  Number of documents: {test_stats['num_documents']}")
print(f"  Sentiment distribution: {test_stats['sentiment_distribution']}")
print(f"  Average document length: {test_stats['avg_doc_length']:.2f} characters")
print(f"  Average word count: {test_stats['avg_word_count']:.2f} words")

print("\nPreprocessing Steps:")
print("1. Text tokenization using custom tokenizer")
print("2. Removal of stopwords and punctuation")
print("3. Lowercase conversion")
print("4. Special character filtering")
print("5. Creation of three text representations:")
print("   - TF-IDF vectorization (max_features=2000, sublinear_tf=True)")
print("   - Bag of Words vectorization (max_features=2000)")
print("   - Pre-trained GloVe word embeddings (100 dimensions)")
print("6. Optional dimensionality reduction using TruncatedSVD")

# Optional: Add more statistics about tokenized data
print("\nCalculating statistics on tokenized data...")
# Create a sample of tokenized documents using your tokenize function
# This assumes you've already defined your tokenize function
from nltk.tokenize import sent_tokenize, word_tokenize
import string
from nltk.corpus import stopwords

# Use a sample for faster processing
sample_data = train_data.sample(min(1000, len(train_data)), random_state=42)
tokenized_samples = tokenize(sample_data['text'])

# Calculate tokenized text statistics
tokenized_lengths = [len(doc) for doc in tokenized_samples]
avg_tokenized_length = sum(tokenized_lengths) / len(tokenized_lengths) if tokenized_lengths else 0
tokenized_vocab = set()
for doc in tokenized_samples:
    tokenized_vocab.update(doc)

print(f"After tokenization (based on sample of {len(tokenized_samples)} documents):")
print(f"  Average tokens per document: {avg_tokenized_length:.2f}")
print(f"  Vocabulary size after preprocessing: {len(tokenized_vocab)} unique tokens")
print(f"  Vocabulary reduction: {100 - len(tokenized_vocab)/train_stats['vocab_size']*100:.2f}% reduction from raw text")

In [ ]:
def preprocess_data(train_data, test_data,
                   representation='bow',
                   features_number=2000,
                   reduce_features=False,
                   reduced_features_number=None):
    # Split training data
    X_train, X_val, y_train, y_val = train_test_split(
        train_data['text'], train_data['sentiment'],
        test_size=0.3, random_state=42)
    X_test, y_test = test_data['text'], test_data['sentiment']

    # Create label mapping
    unique_labels = pd.concat([y_train, y_val, y_test]).unique()
    label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}

    # Convert labels to indices
    y_train = np.array([label_to_idx[label] for label in y_train], dtype=np.int64)
    y_val = np.array([label_to_idx[label] for label in y_val], dtype=np.int64)
    y_test = np.array([label_to_idx[label] for label in y_test], dtype=np.int64)

    # Tokenize texts
    X_train_tokenized = tokenize(X_train)
    X_val_tokenized = tokenize(X_val)
    X_test_tokenized = tokenize(X_test)

    # Convert tokenized texts to joined strings
    X_train_joined = [" ".join(x) for x in X_train_tokenized]
    X_val_joined = [" ".join(x) for x in X_val_tokenized]
    X_test_joined = [" ".join(x) for x in X_test_tokenized]

    # Select vectorizer based on representation type
    if representation == "tfidf":
        vectorizer = TfidfVectorizer(max_features=features_number, sublinear_tf=True)
        X_train = vectorizer.fit_transform(X_train_joined).toarray()
        X_val = vectorizer.transform(X_val_joined).toarray()
        X_test = vectorizer.transform(X_test_joined).toarray()
        feature_names = vectorizer.get_feature_names_out()

    elif representation == "embeddings":
        print("\nLoading pre-trained word embeddings...")
        word_vectors = api.load("glove-wiki-gigaword-100")  # 100-dimensional GloVe embeddings
        print(f"Loaded {len(word_vectors.key_to_index)} word vectors with dimension {word_vectors.vector_size}")

        # Process word embeddings for each dataset
        X_train, X_val, X_test = [], [], []

        print("Processing training embeddings...")
        for doc in X_train_tokenized:
            embeddings = [word_vectors[word] for word in doc if word in word_vectors]
            centroid = np.mean(embeddings, axis=0) if embeddings else np.zeros(word_vectors.vector_size)
            X_train.append(centroid)

        print("Processing validation embeddings...")
        for doc in X_val_tokenized:
            embeddings = [word_vectors[word] for word in doc if word in word_vectors]
            centroid = np.mean(embeddings, axis=0) if embeddings else np.zeros(word_vectors.vector_size)
            X_val.append(centroid)

        print("Processing test embeddings...")
        for doc in X_test_tokenized:
            embeddings = [word_vectors[word] for word in doc if word in word_vectors]
            centroid = np.mean(embeddings, axis=0) if embeddings else np.zeros(word_vectors.vector_size)
            X_test.append(centroid)

        X_train = np.array(X_train, dtype=np.float32)
        X_val = np.array(X_val, dtype=np.float32)
        X_test = np.array(X_test, dtype=np.float32)
        feature_names = [f"dim_{i}" for i in range(word_vectors.vector_size)]

    else:  # "bow" case
        vectorizer = CountVectorizer(max_features=features_number)
        X_train = vectorizer.fit_transform(X_train_joined).toarray()
        X_val = vectorizer.transform(X_val_joined).toarray()
        X_test = vectorizer.transform(X_test_joined).toarray()
        feature_names = vectorizer.get_feature_names_out()

    # Apply dimensionality reduction if requested
    if reduce_features and reduced_features_number is not None:
        svd = TruncatedSVD(n_components=reduced_features_number, random_state=42)
        X_train = svd.fit_transform(X_train)
        X_val = svd.transform(X_val)
        X_test = svd.transform(X_test)
        feature_names = [f"component_{i}" for i in range(reduced_features_number)]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), feature_names, label_to_idx

## Baseline MLP model
The `BaselineSentimentClassifier` works on a simple principle: words tend to be associated with specific sentiments. During training it:
  
  1. Identifes the most common sentiment in the dataset as a fallback
  2. For each word, counts how often it appears with each sentiment
  3. Assigns each word the sentiment it most frequently appears with

When predicting, the classifier:

  1. Identifies which known words are present in the text
  2. Looks up the pre-determined sentiment for each word
  3. Predicts the majority sentiment among these words
  4. Falls back to the most common sentiment of no known words are found

This approach is simple and interpretable but ignores word order and context.

In [ ]:
class BaselineSentimentClassifier:
    def __init__(self):
        self.word_to_sentiment = {}
        self.most_common_sentiment = None
        self.feature_names = None

    """
        Train the baseline model
        X_train: sparse matrix or array of shape (n_samples, n_features)
        y_train: array of shape (n_samples)
        feature_names: list of feature names (words/tokens)
    """
    def train(self, X_train, y_train, feature_names=None):
        self.feature_names = feature_names

        # Get most common sentiment label overall (fallback)
        sentiment_counts = Counter(y_train)
        self.most_common_sentiment = sentiment_counts.most_common(1)[0][0]

        # For each feature (word), find most frequently associated sentiment
        if self.feature_names is not None:
            for i, feature in enumerate(self.feature_names):
                # Get feature column
                feature_col = X_train[:, i].toarray().flatten() if hasattr(X_train, 'toarray') else X_train[:, i]

                # Count sentiment occurrences when feature is present
                sentiment_counts = defaultdict(int)
                for sample_idx, value in enumerate(feature_col):
                    if value > 0:  # Feature is present in this sample
                        sentiment_counts[y_train[sample_idx]] += 1

                # Assign most common sentiment for this feature
                if sentiment_counts:
                    self.word_to_sentiment[feature] = max(sentiment_counts.items(), key=lambda x: x[1])[0]

    """
      Predict sentiment based on most common sentiment per feature
      X_test: sparse matrix or array of shape (n_samples, n_features)
    """
    def predict(self, X_test):
        predictions = []

        # If we don't have feature names, just predict most common sentiment
        if self.feature_names is None:
            return np.full(X_test.shape[0], self.most_common_sentiment)

        for i in range(X_test.shape[0]):
            # Get sample features
            sample = X_test[i].toarray().flatten() if hasattr(X_test, 'toarray') else X_test[i]

            # Get sentiments for present features
            sample_sentiments = []
            for feature_idx, value in enumerate(sample):
                if value > 0 and self.feature_names[feature_idx] in self.word_to_sentiment:
                    sample_sentiments.append(self.word_to_sentiment[self.feature_names[feature_idx]])

            # Predict most common sentiment from present features, or fallback
            if sample_sentiments:
                predictions.append(Counter(sample_sentiments).most_common(1)[0][0])
            else:
                predictions.append(self.most_common_sentiment)

        return np.array(predictions)

    """
      Create pseudo-probabilities for compatibility with evaluation metrics
      X_test: sparse matrix or array of shape (n_samples, n_features)
    """
    def predict_proba(self, X_test):
        predictions = self.predict(X_test)
        n_classes = len(np.unique(predictions))

        # Create one-hot encoded pseudo-probabilities
        probs = np.zeros((X_test.shape[0], n_classes))
        for i, pred in enumerate(predictions):
            probs[i, pred] = 1.0

        return probs

In [ ]:
# Method that evaluates baseline model on all data splits
def evaluate_baseline(baseline_model, X_train, y_train, X_val, y_val, X_test, y_test):
    # Make predictions
    train_preds = baseline_model.predict(X_train)
    val_preds = baseline_model.predict(X_val)
    test_preds = baseline_model.predict(X_test)

    # Get "probabilities" (for consistency with other models)
    train_probs = baseline_model.predict_proba(X_train)
    val_probs = baseline_model.predict_proba(X_val)
    test_probs = baseline_model.predict_proba(X_test)

    # Calculate metrics
    train_metrics = calculate_metrics(y_train, train_preds, train_probs)
    val_metrics = calculate_metrics(y_val, val_preds, val_probs)
    test_metrics = calculate_metrics(y_test, test_preds, test_probs)

    return {
        'train': train_metrics,
        'val': val_metrics,
        'test': test_metrics
    }

# Method that calculate consistent metrics for all models
def calculate_metrics(y_true, y_pred, y_prob):
    n_classes = y_prob.shape[1]

    # Overall accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # Per-class metrics
    class_metrics = {}
    pr_auc_scores = []

    for i in range(n_classes):
        # Convert to binary classification problem (one-vs-rest)
        y_true_binary = (y_true == i).astype(int)
        y_pred_binary = (y_pred == i).astype(int)
        y_prob_class = y_prob[:, i]

        # Calculate precision, recall, F1
        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)

        # Calculate precision-recall curve and AUC
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_binary, y_prob_class)
        pr_auc = auc(recall_curve, precision_curve)
        pr_auc_scores.append(pr_auc)

        # Store metrics
        class_metrics[i] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'pr_auc': pr_auc
        }

    # Macro-averaged metrics
    macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    macro_pr_auc = np.mean(pr_auc_scores)

    return {
        'accuracy': accuracy,
        'class_metrics': class_metrics,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'macro_pr_auc': macro_pr_auc
    }

## Sentiment MLP model
The `SentimentMLP` is a Multi-Layer Perceptron that can learn complex patterns in text data:
  1. **Structure**: Input Layer → Hidden layer(s) → Output layer
    * Input layer receives numerical text features (embeddings)
    * Hidden layers learn increasingly complex patterns
    * Output layer produces sentiment predictions
  2. **Key Components**:
    * ReLU activations introduce non-linearity
    * Dropout (default 0.5) prevents overfitting by randomly deactivating neurons
    * Optional batch/layer normalization stabilizes training
  3. **Advantages over Baseline**:
    * Can learn non-linear relationships between words and sentiment
    * Considers word combinations rather than treating each word independently
    * Handles context better (e.g. can learn that "not good" is negative)
    * Generally achieves higher accuracy, especially on complex texts

In [ ]:
class SentimentMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout=0.5, batch_norm=False,
                 layer_norm=False, classes_number=2):
        super(SentimentMLP, self).__init__()

        # Create a list to hold all layers
        layers = []

        # Input layer
        prev_dim = input_dim

        # Add hidden layers
        for hidden_dim in hidden_layers:
            # Linear layer
            layers.append(nn.Linear(prev_dim, hidden_dim))

            # Batch normalization (optional)
            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))

            # Layer normalization (optional)
            if layer_norm:
                layers.append(nn.LayerNorm(hidden_dim))

            # Activation function
            layers.append(nn.ReLU())

            # Dropout for regularization
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            prev_dim = hidden_dim

        # Output layer
        layers.append(nn.Linear(prev_dim, classes_number))

        # Combine all layers
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
# This is an enhances training function with early stopping, comprehensive metrics, and plotting
def train_model(model, train_loader, val_loader, test_loader, criterion, optimizer, epochs=20, device='cuda', patience=5):
    # Initialize variables for early stopping
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_path = "best_sentiment_mlp_model.pt"

    # Lists to store metrics for plotting
    train_losses = []
    val_losses = []

    # Training loop
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} (Training)"):
            # Data already moved to device in DataLoader initialization

            # Forward pass
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

        # Calculate average training loss
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} (Validation)"):
                # Data already moved to device in DataLoader initialization

                # Forward pass
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)

                val_loss += loss.item() * X_batch.size(0)

        # Calculate average validation loss
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        # Print epoch results
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.5f}, Val Loss: {val_loss:.5f}")

        # Early stopping and model saving
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            # Save the best model
            torch.save(model.state_dict(), best_model_path)
            print(f"Model improved - saving to {best_model_path}")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{patience}")

            if patience_counter >= patience:
                print(f"Early stopping after {epoch+1} epochs")
                break

    # Load the best model
    model.load_state_dict(torch.load(best_model_path))

    return model, train_losses, val_losses

# Method that gets model predictions, predicted probabilities, and true labels from a data loader
def get_predictions_and_labels(model, data_loader):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            # Forward pass
            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)

            # Store results
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    return np.array(all_preds), np.array(all_probs), np.array(all_labels)

# Method that computes metric per-class and macro-averaged metrics
def compute_class_metrics(y_true, y_pred, y_prob):
    n_classes = y_prob.shape[1]

    # Initialize dictionaries to store metrics
    class_metrics = {}
    for i in range(n_classes):
        class_metrics[i] = {}

        # Convert to binary classification problem (one-vs-rest)
        y_true_binary = (y_true == i).astype(int)
        y_pred_binary = (y_pred == i).astype(int)
        y_prob_class = y_prob[:, i]

        # Calculate precision, recall, F1
        class_metrics[i]['precision'] = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        class_metrics[i]['recall'] = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        class_metrics[i]['f1'] = f1_score(y_true_binary, y_pred_binary, zero_division=0)

        # Calculate precision-recall curve and AUC
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_binary, y_prob_class)
        class_metrics[i]['pr_auc'] = auc(recall_curve, precision_curve)

    # Calculate macro-averaged metrics
    macro_metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': np.mean([class_metrics[i]['precision'] for i in range(n_classes)]),
        'macro_recall': np.mean([class_metrics[i]['recall'] for i in range(n_classes)]),
        'macro_f1': np.mean([class_metrics[i]['f1'] for i in range(n_classes)]),
        'macro_pr_auc': np.mean([class_metrics[i]['pr_auc'] for i in range(n_classes)])
    }

    return class_metrics, macro_metrics

# Comprehendive evaluation method with detailed metrics for all data splits
def evaluate_model_detailed(model, train_loader, val_loader, test_loader):
    # Get predictions for all splits
    train_preds, train_probs, train_labels = get_predictions_and_labels(model, train_loader)
    val_preds, val_probs, val_labels = get_predictions_and_labels(model, val_loader)
    test_preds, test_probs, test_labels = get_predictions_and_labels(model, test_loader)

    # Compute metrics for all splits
    train_class_metrics, train_macro = compute_class_metrics(train_labels, train_preds, train_probs)
    val_class_metrics, val_macro = compute_class_metrics(val_labels, val_preds, val_probs)
    test_class_metrics, test_macro = compute_class_metrics(test_labels, test_preds, test_probs)

    # Get number of classes
    n_classes = train_probs.shape[1]

    # Print per-class metrics
    print("\n=== PER-CLASS METRICS ===")
    print(f"{'Class':<10}{'Split':<10}{'Precision':<12}{'Recall':<12}{'F1':<12}{'PR-AUC':<12}")
    print("-" * 65)

    for i in range(n_classes):
        print(f"{i:<10}{'Train':<10}{train_class_metrics[i]['precision']:.4f}{' '*8}{train_class_metrics[i]['recall']:.4f}{' '*8}{train_class_metrics[i]['f1']:.4f}{' '*8}{train_class_metrics[i]['pr_auc']:.4f}")
        print(f"{i:<10}{'Dev':<10}{val_class_metrics[i]['precision']:.4f}{' '*8}{val_class_metrics[i]['recall']:.4f}{' '*8}{val_class_metrics[i]['f1']:.4f}{' '*8}{val_class_metrics[i]['pr_auc']:.4f}")
        print(f"{i:<10}{'Test':<10}{test_class_metrics[i]['precision']:.4f}{' '*8}{test_class_metrics[i]['recall']:.4f}{' '*8}{test_class_metrics[i]['f1']:.4f}{' '*8}{test_class_metrics[i]['pr_auc']:.4f}")
        print("-" * 65)

    # Print macro-averaged metrics
    print("\n=== MACRO-AVERAGED METRICS ===")
    print(f"{'Split':<10}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'PR-AUC':<12}")
    print("-" * 65)
    print(f"{'Train':<10}{train_macro['accuracy']:.4f}{' '*8}{train_macro['macro_precision']:.4f}{' '*8}{train_macro['macro_recall']:.4f}{' '*8}{train_macro['macro_f1']:.4f}{' '*8}{train_macro['macro_pr_auc']:.4f}")
    print(f"{'Dev':<10}{val_macro['accuracy']:.4f}{' '*8}{val_macro['macro_precision']:.4f}{' '*8}{val_macro['macro_recall']:.4f}{' '*8}{val_macro['macro_f1']:.4f}{' '*8}{val_macro['macro_pr_auc']:.4f}")
    print(f"{'Test':<10}{test_macro['accuracy']:.4f}{' '*8}{test_macro['macro_precision']:.4f}{' '*8}{test_macro['macro_recall']:.4f}{' '*8}{test_macro['macro_f1']:.4f}{' '*8}{test_macro['macro_pr_auc']:.4f}")

    # Plot precision-recall curves for each class
    plot_precision_recall_curves(train_labels, train_probs, val_labels, val_probs, test_labels, test_probs, n_classes)

    # Return all metrics for possible further use
    return {
        'train': {'class': train_class_metrics, 'macro': train_macro},
        'val': {'class': val_class_metrics, 'macro': val_macro},
        'test': {'class': test_class_metrics, 'macro': test_macro}
    }

# Method that plots training and validations loss curves
def plot_loss_curves(train_losses, val_losses, title="MLP Classifier"):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss', linestyle='--')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(f'{title} - Training and Validation Loss Curves')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig('loss_curves.png')
    plt.show()

# Method that plots precision-recall curves for each class and data split
def plot_precision_recall_curves(train_labels, train_probs, val_labels, val_probs, test_labels, test_probs, n_classes):
    # Set up the plot grid
    fig, axes = plt.subplots(n_classes, 3, figsize=(18, 5 * n_classes))
    if n_classes == 1:
        axes = axes.reshape(1, 3)

    splits = ['Train', 'Validation', 'Test']
    colors = ['blue', 'green', 'red']

    for class_idx in range(n_classes):
        for split_idx, (split, labels, probs) in enumerate(zip(
            splits,
            [train_labels, val_labels, test_labels],
            [train_probs, val_probs, test_probs]
        )):
            # Convert to binary classification for the current class
            y_true_binary = (labels == class_idx).astype(int)
            y_prob_class = probs[:, class_idx]

            # Calculate precision-recall curve
            precision, recall, _ = precision_recall_curve(y_true_binary, y_prob_class)
            pr_auc = auc(recall, precision)

            # Plot the curve
            ax = axes[class_idx, split_idx]
            ax.plot(recall, precision, color=colors[split_idx], lw=2,
                   label=f'PR-AUC: {pr_auc:.4f}')
            ax.set_xlabel('Recall')
            ax.set_ylabel('Precision')
            ax.set_title(f'Class {class_idx} - {split} Precision-Recall Curve')
            ax.legend(loc='best')
            ax.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig('precision_recall_curves.png')
    plt.show()

We use `Hyperaparameter tuning` to find the best settings for our model.

The method tries different combinations of settings to find what works best:
```
param_grid = {
    'hidden_layers': [[64, 32], [128, 64], [256, 128], [128, 64, 32]],
    'learning_rate': [1e-4, 1e-3, 5e-3],
    'batch_size': [32, 64, 128],
    'dropout': [0.0, 0.2, 0.5],
    'batch_norm': [False, True],
}
```
These settings control:
  * **hiden_layers**: The network's structure
  * **learning_rate**: How quickly the model learns
  * **batch_size**: How many examples to process at once
  * **dropout**: Technique to prevent overfitting
  * **batch_norm**: Helps training stability


For each possible combination:
  1. Builds a model with those settings
  2. Trains it briefly (10 epochs)
  3. Tests how well it works on validation data
  4. Keeps track of the best poerformance

After trying all combinations:
  1. Sorts results by performance
  2. SHows top 3 best configurations
  3. Returns the best model and its settings

In [ ]:
def hyperparameter_tuning(X_train, y_train, X_val, y_val, device='cuda', verbose=True):
    # Define hyperparameter grid to search
    param_grid = {
        'hidden_layers': [
            [64, 32],
            [128, 64],
            [256, 128],
            [128, 64, 32]
        ],
        'learning_rate': [1e-4, 1e-3, 5e-3],
        'batch_size': [32, 64, 128],
        'dropout': [0.0, 0.2, 0.5],
        'batch_norm': [False, True],
    }

    # Generate all combinations of hyperparameters
    params_list = list(ParameterGrid(param_grid))
    if verbose:
        print(f"Testing {len(params_list)} hyperparameter combinations")

    # Track results
    results = []
    best_val_loss = float('inf')
    best_model = None
    best_params = None

    # Test each combination
    for params in tqdm(params_list, desc="Hyperparameter Tuning"):
        if verbose:
            print(f"\nTesting parameters: {params}")

        # Prepare data loaders with current batch size
        batch_size = params['batch_size']
        train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(y_train).to(device))
        val_dataset = TensorDataset(torch.FloatTensor(X_val).to(device), torch.LongTensor(y_val).to(device))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

        # Create model with current parameters
        input_dim = X_train.shape[1]
        model = SentimentMLP(
            input_dim,
            hidden_layers=params['hidden_layers'],
            dropout=params['dropout'],
            batch_norm=params['batch_norm'],
            layer_norm=False,
            classes_number=len(np.unique(y_train))
        ).float().to(device)

        # Setup training
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])

        # Quick training for hyperparameter search (10 epochs)
        train_losses = []
        val_losses = []
        epochs = 10  # Reduced epochs for quick search

        for epoch in range(epochs):
            # Training phase
            model.train()
            train_loss = 0.0
            for X_batch, y_batch in train_loader:
                # Forward pass
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)

                # Backward pass and optimization
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                train_loss += loss.item() * X_batch.size(0)

            # Calculate average training loss
            train_loss /= len(train_loader.dataset)
            train_losses.append(train_loss)

            # Validation phase
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    # Forward pass
                    y_pred = model(X_batch)
                    loss = criterion(y_pred, y_batch)

                    val_loss += loss.item() * X_batch.size(0)

            # Calculate average validation loss
            val_loss /= len(val_loader.dataset)
            val_losses.append(val_loss)

            if verbose and epoch % 5 == 0:
                print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.5f}, Val Loss: {val_loss:.5f}")

        # Store best model
        final_val_loss = val_losses[-1]
        results.append((params, final_val_loss))

        if final_val_loss < best_val_loss:
            best_val_loss = final_val_loss
            best_model = copy.deepcopy(model)
            best_params = params
            if verbose:
                print(f"New best model found! Validation loss: {best_val_loss:.5f}")

    # Sort results by validation loss
    results.sort(key=lambda x: x[1])

    # Print top 3 configurations
    if verbose:
        print("\nTop 3 hyperparameter configurations:")
        for i, (params, val_loss) in enumerate(results[:3]):
            print(f"{i+1}. Val Loss: {val_loss:.5f}, Params: {params}")

    return best_model, best_params, results

## Training of both models

### Training using **TfIdf vectorizer** embeddings

We firstly split the data and use the chosen vectorizer to create the embeddings

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test), feature_names, label_to_idx = preprocess_data(
    train_data, test_data, representation="tfidf"
)

We initialize the baseline model

In [ ]:
baseline_model = BaselineSentimentClassifier()
baseline_model.train(X_train, y_train, feature_names)


In [ ]:
baseline_metrics = evaluate_baseline(baseline_model, X_train, y_train, X_val, y_val, X_test, y_test)

print("\n=== BASELINE MODEL METRICS ===")
print(f"Accuracy: {baseline_metrics['test']['accuracy']:.4f}")
print(f"Macro F1: {baseline_metrics['test']['macro_f1']:.4f}")
print(f"Macro PR-AUC: {baseline_metrics['test']['macro_pr_auc']:.4f}")

**Observations**: Our results show our simple-word-counting baseline model performs suprisingly well. The accuracy shows that it correctly classifies about 4 out of 5 texts. The identical F1 score suggests balanced performance across sentiment classes. The higher PR-AUC indicates good confidence in its predictions

For a baseline that just counts word-sentiment associations without understanding context, there are solid results. However, there's still room for improvement with more sophisticated models.

We tune the hyper parameters to create the best possible MLP model

In [ ]:
# Tune hyperparameters
best_model, best_params, tuning_results = hyperparameter_tuning(
    X_train, y_train, X_val, y_val, device=device
)

print(f"Best parameters: {best_params}")

In [ ]:
batch_size = best_params['batch_size']  # Use the best batch size

train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(y_train).to(device))
val_dataset = TensorDataset(torch.FloatTensor(X_val).to(device), torch.LongTensor(y_val).to(device))
test_dataset = TensorDataset(torch.FloatTensor(X_test).to(device), torch.LongTensor(y_test).to(device))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

input_dim = X_train.shape[1]

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(best_model.parameters(), lr=best_params['learning_rate'])


In [ ]:
# Train the model with the enhanced function
model, train_losses, val_losses = train_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,  # Added test_loader
    criterion=criterion,
    optimizer=optimizer,
    epochs=20,
    device=device,
    patience=5
)

In [ ]:
# Plot loss curves
plot_loss_curves(train_losses, val_losses, title="Sentiment MLP Classifier")

# Get detailed evaluation metrics
all_metrics = evaluate_model_detailed(model, train_loader, val_loader, test_loader)


**Observations**:
  * The neural network model substantially outperforms the baseline, showing why more complex models are often worth the investment.
  * The model shows similar effecetiveness form both positive and negative sentiment detection, avoiding bias toward either class.
  * The reasonable gap between training and testing performance indicates the model learned meaningful patterns rather than just memorizing the training data.
  * High PR-AUC scores across all data splits suggest the model makes predictions with strong confidence.
  * The balanced precision and recall metrics show the model doesn't sacrifice one aspect of performance for another.


The neural network demonstrates clear advantages over the simples word-counting approach while maintaining consistent performance across different evaluation metrics. This confirms that the additional complexity of the MLP architecture translates to meaningful improvements in sentiment analysis accuracy.

In [ ]:
# Create the comparison structures
models_comparison = {
    'Baseline': baseline_metrics,
    'MLP Classifier': all_metrics
}

# Print comparison table for test set
print("\n=== MODEL COMPARISON (TEST SET) ===")
print(f"{'Model':<20}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'PR-AUC':<12}")
print("-" * 75)

for model_name, metrics in models_comparison.items():
    # Access test metrics based on the different structures
    if model_name == 'Baseline':
        test_metrics = metrics['test']
    else:  # MLP Classifier
        test_metrics = metrics['test']['macro']  # The MLP metrics have a nested 'macro' key

    print(f"{model_name:<20}{test_metrics['accuracy']:.4f}{' '*8}{test_metrics['macro_precision']:.4f}{' '*8}{test_metrics['macro_recall']:.4f}{' '*8}{test_metrics['macro_f1']:.4f}{' '*8}{test_metrics['macro_pr_auc']:.4f}")

### Training using **Bag of Words** embeddings

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test), feature_names, label_to_idx = preprocess_data(
    train_data, test_data, representation="bow"
)

In [ ]:
baseline_model = BaselineSentimentClassifier()
baseline_model.train(X_train, y_train, feature_names)

In [ ]:
baseline_metrics = evaluate_baseline(baseline_model, X_train, y_train, X_val, y_val, X_test, y_test)

print("\n=== BASELINE MODEL METRICS ===")
print(f"Accuracy: {baseline_metrics['test']['accuracy']:.4f}")
print(f"Macro F1: {baseline_metrics['test']['macro_f1']:.4f}")
print(f"Macro PR-AUC: {baseline_metrics['test']['macro_pr_auc']:.4f}")

In [ ]:
# Tune hyperparameters
best_model, best_params, tuning_results = hyperparameter_tuning(
    X_train, y_train, X_val, y_val, device=device
)

print(f"Best parameters: {best_params}")

In [ ]:
batch_size = best_params['batch_size']  # Use the best batch size

train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(y_train).to(device))
val_dataset = TensorDataset(torch.FloatTensor(X_val).to(device), torch.LongTensor(y_val).to(device))
test_dataset = TensorDataset(torch.FloatTensor(X_test).to(device), torch.LongTensor(y_test).to(device))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

input_dim = X_train.shape[1]

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(best_model.parameters(), lr=best_params['learning_rate'])

# Train the model with the enhanced function
model, train_losses, val_losses = train_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,  # Added test_loader
    criterion=criterion,
    optimizer=optimizer,
    epochs=20,
    device=device,
    patience=5
)


In [ ]:
# Plot loss curves
plot_loss_curves(train_losses, val_losses, title="Sentiment MLP Classifier")

# Get detailed evaluation metrics
all_metrics = evaluate_model_detailed(model, train_loader, val_loader, test_loader)


**Observations**: The `Bag of Words (BoW)` approach with the MLP classifier shows several interesting patterns:
  1. The model demonstrates excellent performance on the training data, suggesting it effectively captures the patters present in the training examples.
  2. There's a noticeable difference between training and test performance, indicating some degree of overfitting to the training data.
  3. Similar to the previous model, this approach maintains balances performance across both sentiment classes, showing no bias toward positive or negative sentiment detection.
  4. This BoW approach performs slightly better then the TfIdf vectorizer apporach, suggesting that the feature representation matters as much as the model architecture.

In [ ]:
# Create the comparison structures
models_comparison = {
    'Baseline': baseline_metrics,
    'MLP Classifier': all_metrics
}

# Print comparison table for test set
print("\n=== MODEL COMPARISON (TEST SET) ===")
print(f"{'Model':<20}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'PR-AUC':<12}")
print("-" * 75)

for model_name, metrics in models_comparison.items():
    # Access test metrics based on the different structures
    if model_name == 'Baseline':
        test_metrics = metrics['test']
    else:  # MLP Classifier
        test_metrics = metrics['test']['macro']  # The MLP metrics have a nested 'macro' key

    print(f"{model_name:<20}{test_metrics['accuracy']:.4f}{' '*8}{test_metrics['macro_precision']:.4f}{' '*8}{test_metrics['macro_recall']:.4f}{' '*8}{test_metrics['macro_f1']:.4f}{' '*8}{test_metrics['macro_pr_auc']:.4f}")


### Training using **pre-trained word embeddings** embeddings

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test), feature_names, label_to_idx = preprocess_data(
    train_data, test_data, representation="embeddings"
)

In [ ]:
baseline_model = BaselineSentimentClassifier()
baseline_model.train(X_train, y_train, feature_names)

In [ ]:
baseline_metrics = evaluate_baseline(baseline_model, X_train, y_train, X_val, y_val, X_test, y_test)

print("\n=== BASELINE MODEL METRICS ===")
print(f"Accuracy: {baseline_metrics['test']['accuracy']:.4f}")
print(f"Macro F1: {baseline_metrics['test']['macro_f1']:.4f}")
print(f"Macro PR-AUC: {baseline_metrics['test']['macro_pr_auc']:.4f}")

In [ ]:
# Tune hyperparameters
best_model, best_params, tuning_results = hyperparameter_tuning(
    X_train, y_train, X_val, y_val, device=device
)

print(f"Best parameters: {best_params}")

In [ ]:
batch_size = best_params['batch_size']  # Use the best batch size

train_dataset = TensorDataset(torch.FloatTensor(X_train).to(device), torch.LongTensor(y_train).to(device))
val_dataset = TensorDataset(torch.FloatTensor(X_val).to(device), torch.LongTensor(y_val).to(device))
test_dataset = TensorDataset(torch.FloatTensor(X_test).to(device), torch.LongTensor(y_test).to(device))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

input_dim = X_train.shape[1]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(best_model.parameters(), lr=best_params['learning_rate'])

# Train the model with the enhanced function
model, train_losses, val_losses = train_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,  # Added test_loader
    criterion=criterion,
    optimizer=optimizer,
    epochs=20,
    device=device,
    patience=5
)

In [ ]:
# Plot loss curves
plot_loss_curves(train_losses, val_losses, title="Sentiment MLP Classifier")

# Get detailed evaluation metrics
all_metrics = evaluate_model_detailed(model, train_loader, val_loader, test_loader)

**Observations**: The model using pre-trained word embeddings shows several interesting patters compared to previous approaches:
  1. Surprisingly, the pre-trained embeddings model performs noticeably worse than both the simpler Bag of Words approach and the TfIdf vectorizer implementation.
  2. There's a slight imbalance in performance between the two sentiment classes, with negative sentiment class showing somewhat better recall than positive sentiment.
  3. The difference between training and test performance is less dramatic than with the Bag of Words model, suggesting better generalization despite lower overall performance.
  4. The performance drop is consistent across all metrics, indicating fundamental limitation rather than a problem with specific aspects of classification.

In [ ]:
# Create the comparison structures
models_comparison = {
    'Baseline': baseline_metrics,
    'MLP Classifier': all_metrics
}

# Print comparison table for test set
print("\n=== MODEL COMPARISON (TEST SET) ===")
print(f"{'Model':<20}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'PR-AUC':<12}")
print("-" * 75)

for model_name, metrics in models_comparison.items():
    # Access test metrics based on the different structures
    if model_name == 'Baseline':
        test_metrics = metrics['test']
    else:  # MLP Classifier
        test_metrics = metrics['test']['macro']  # The MLP metrics have a nested 'macro' key

    print(f"{model_name:<20}{test_metrics['accuracy']:.4f}{' '*8}{test_metrics['macro_precision']:.4f}{' '*8}{test_metrics['macro_recall']:.4f}{' '*8}{test_metrics['macro_f1']:.4f}{' '*8}{test_metrics['macro_pr_auc']:.4f}")
